In [1]:
import pandas as pd
import plotly.express as px

In [2]:
df = pd.read_csv('/Users/anandguntuku/Movies/VSCodeFolder/SarvaniMLOps/Data/merged_df.csv')

/var/folders/_n/g_d34gh57sb9jndsrxxvvdy40000gn/T/ipykernel_2013/1756544048.py:1: DtypeWarning: Columns (28) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('/Users/anandguntuku/Movies/VSCodeFolder/SarvaniMLOps/Data/merged_df.csv')


In [3]:
pd.set_option('display.max_columns', 100)
df.head(2)

,Outlet,Date_Time,Bill_No,Item_Price,Item_Quantity,Item_Total_Amount,Bill_Item_Count,Order_Type,Weekday,Total_Bill_Amount_Corrected,Daily_Bill_Amount,Corrected_Item_Name,Portion_Code,Corrected_Item_Category,total_weight_kg,Festivity_x,Bill_Class,Date,Unnamed: 0,Year,Month,Week_Number,Day,Festivity_y,Quarter,Festivity_Group,Day_Type,DayFestGroup,Holiday_Name
0,MADDILAPALEM OUTLET,2022-04-01 06:54:17.779,2022211,266.6667,1.0,279.9967,1,general,4,280.0,209884.0,KOVABISCUITS(ORANGE)0500G,0500G,KOVA SWEETS0500G,0.5,regular,Small,2022-04-01,33,2022,4,13,1,regular,Q1,Non-Festive,Wknd,Wknd - Non-Festive,NaN
1,MADDILAPALEM OUTLET,2022-04-01 08:27:52.593,2022212,180.9524,1.0,190.0024,1,general,4,190.0,209884.0,MOTHICHOORLADDURED0500G,0500G,LADDUS0500G,0.5,regular,Micro,2022-04-01,34,2022,4,13,1,regular,Q1,Non-Festive,Wknd,Wknd - Non-Festive,NaN


In [4]:
df_hol = pd.read_csv('/Users/anandguntuku/Movies/VSCodeFolder/SarvaniMLOps/Data/Holiday_Data copy.csv')
df_hol

,Date,Holiday
0,2026-01-01,New Year
1,2025-12-31,New Year
2,2025-12-25,Christmas
3,2025-12-24,Christmas
4,2025-10-20,Diwali
...,...,...
184,2020-01-15,Sankranti
185,2020-01-14,Sankranti
186,2020-01-13,Sankranti
187,2020-01-01,New Year


In [5]:
import pandas as pd

def compute_weekly_sales_summary(df,df_hol):
    df = df.copy()
    print(df.shape, "Shape of initial table")
    # Ensure correct dtypes
    df['Date'] = pd.to_datetime(df['Date'])
    df['Week'] = df['Date'] - pd.to_timedelta(df['Date'].dt.weekday, unit='D')  # Week starting Monday

    # Clean item name
    df['Item_Name'] = df['Corrected_Item_Name'].str.replace(r'\d+G', '', regex=True).str.replace(r'[_\-\s]+', ' ').str.strip()

    # Compute price per kg
    df['price_perkg'] = df['Item_Total_Amount'] / df['total_weight_kg']
    df = df[df['total_weight_kg'] > 0]  # avoid division by zero

    # Group by week-item-outlet
    grouped = df.groupby(['Item_Name', 'Outlet', 'Week', 'Corrected_Item_Name', 'DayFestGroup'], as_index=False).agg({
        'total_weight_kg': 'sum',
        'Item_Total_Amount': 'sum',
        'price_perkg': 'mean',
        'Date': 'max'
    })

    grouped = grouped.rename(columns={
        'Date': 'Date',
        'price_perkg': 'price_perkg',
        'total_weight_kg': 'Weekly_Weight',
        'Item_Total_Amount': 'Weekly_Amount',
        'DayFestGroup':'DayFestGroup'
    })

    grouped.reset_index(inplace=True)
    grouped.rename(columns={'index': 'Unnamed: 0'}, inplace=True)

    # Reorder columns
    final = grouped[['Date', 'Item_Name','Corrected_Item_Name', 'Outlet', 'price_perkg', 
                     'Week', 'Weekly_Weight', 'Weekly_Amount', 'DayFestGroup']]

    print(final.shape, "Shape of intermediate table")
    # Ensure date columns are datetime
    final['Date'] = pd.to_datetime(final['Date'])
    final['Week'] = pd.to_datetime(final['Week'])  # Week should already be start of week (Monday)
    df_hol['Date'] = pd.to_datetime(df_hol['Date'])

    # Step 1: Map each holiday to its week (Monday of that week)
    df_hol['Week'] = df_hol['Date'] - pd.to_timedelta(df_hol['Date'].dt.weekday, unit='D')

    # Step 2: Find the mode of holiday per week
    week_holiday_mode = (
        df_hol.groupby('Week')['Holiday']
        .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
        .reset_index()
        .rename(columns={'Holiday': 'Event'})
    )

    # Step 3: Merge into df_main on Week
    df1 = final.merge(week_holiday_mode, on='Week', how='left')

    # Step 4: Fill missing weeks with 'No Event'
    df1['Event'] = df1['Event'].fillna('No Event')


    print(df1.shape, "Shape of final table")
    return df1


In [6]:
weekly_sales_table = compute_weekly_sales_summary(df,df_hol)
weekly_sales_table.head(3)

(6407267, 29) Shape of initial table
(781804, 9) Shape of intermediate table


/var/folders/_n/g_d34gh57sb9jndsrxxvvdy40000gn/T/ipykernel_2013/3994150182.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final['Date'] = pd.to_datetime(final['Date'])
/var/folders/_n/g_d34gh57sb9jndsrxxvvdy40000gn/T/ipykernel_2013/3994150182.py:43: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final['Week'] = pd.to_datetime(final['Week'])  # Week should already be start of week (Monday)


(781804, 10) Shape of final table


,Date,Item_Name,Corrected_Item_Name,Outlet,price_perkg,Week,Weekly_Weight,Weekly_Amount,DayFestGroup,Event
0,2021-07-08,AGRAMIXTURE,AGRAMIXTURE0250G,ASILMETTA OUTLET,360.017200,2021-07-05,0.75,270.0129,Wkdy - Non-Festive,No Event
1,2021-07-09,AGRAMIXTURE,AGRAMIXTURE0250G,ASILMETTA OUTLET,359.997200,2021-07-05,0.50,179.9986,Wknd - Non-Festive,No Event
2,2021-08-11,AGRAMIXTURE,AGRAMIXTURE0250G,ASILMETTA OUTLET,360.010533,2021-08-09,1.00,360.0072,Wkdy - Non-Festive,Independence Day


In [7]:
import pandas as pd

def retain_dominant_price_rows(dfx):
    # Ensure Date is in datetime format
    dfx['Date'] = pd.to_datetime(dfx['Date'])

    df_final = []

    for item in dfx['Item_Name'].unique():
        df = dfx[dfx['Item_Name'] == item].copy()
        

        # Extract year-month to group by
        df['Week'] = df['Date'].dt.to_period('W').dt.to_timestamp()

        # Step 1: Find most frequent price_perkg for each month
        mode_price_per_month = (
            df.groupby(['Week'])['price_perkg']
            .agg(lambda x: x.mode().iloc[0])  # If multiple modes, take the first
            .reset_index()
            .rename(columns={'price_perkg': 'mode_price_perkg'})
        )

        # Step 2: Merge back to original data
        df_merged = df.merge(mode_price_per_month, on='Week', how='left')

        # Step 3: Filter only rows where price_perkg == mode
        df_filtered = df_merged[df_merged['price_perkg'] == df_merged['mode_price_perkg']]

        # Optional: Drop helper columns
        df_filtered.drop(columns=['mode_price_perkg'], inplace=True)

        df_final.append(df_filtered)

    return pd.concat(df_final, ignore_index=True)

weekly_sales_mode_price = retain_dominant_price_rows(weekly_sales_table)
weekly_sales_mode_price.head(3)


/var/folders/_n/g_d34gh57sb9jndsrxxvvdy40000gn/T/ipykernel_2013/288989832.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered.drop(columns=['mode_price_perkg'], inplace=True)
/var/folders/_n/g_d34gh57sb9jndsrxxvvdy40000gn/T/ipykernel_2013/288989832.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_filtered.drop(columns=['mode_price_perkg'], inplace=True)
/var/folders/_n/g_d34gh57sb9jndsrxxvvdy40000gn/T/ipykernel_2013/288989832.py:31: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/s

,Date,Item_Name,Corrected_Item_Name,Outlet,price_perkg,Week,Weekly_Weight,Weekly_Amount,DayFestGroup,Event
0,2021-07-08,AGRAMIXTURE,AGRAMIXTURE0250G,ASILMETTA OUTLET,360.0172,2021-07-05,0.75,270.0129,Wkdy - Non-Festive,No Event
1,2021-08-19,AGRAMIXTURE,AGRAMIXTURE0250G,ASILMETTA OUTLET,360.0172,2021-08-16,0.25,90.0043,Wkdy - Non-Festive,Raksha Bandhan
2,2021-08-23,AGRAMIXTURE,AGRAMIXTURE1000G,ASILMETTA OUTLET,359.9971,2021-08-23,1.00,359.9971,Wkdy - Non-Festive,No Event


In [8]:
weekly_sales_mode_price.shape

(300522, 10)

In [9]:
weekly_sales_mode_price['Item_Name'].unique()

array(['AGRAMIXTURE', 'AGRAPETA', 'AGRASEV', 'ALMOND(BADAM)',
       'ALMONDBOUKEJ', 'ALMONDTART', 'ALOOBUJIYA', 'ALOOLAYS',
       'ALOOMASALACHIPS', 'AMLAMINI01PA', 'ANJEERROLL', 'ANJEERSANDWICH',
       'ANNAMAYYALADDU', 'ANZEERTART', 'ARISELUGHEE', 'ATUKULU(KARAM)',
       'ATUKULU(SALT)', 'ATUKULUSEV', 'AYAIBURELU', 'AZMEERKALAKAND',
       'AZMEERKALAKAND(SUGARFREE)', 'BADAMHALWA', 'BADAMLACHHA',
       'BADAMMYSOREPAK', 'BADUSHAGHEE', 'BADUSHAOIL', 'BAKLAVA',
       'BANDARHALWA', 'BANDARLADDU', 'BASANTHI', 'BATHISASOANPAPIDI',
       'BEACHBADAMBURFI', 'BELLAM MITHAI UNDALU', 'BELLAMGAVVALU',
       'BELLAMJELABI', 'BELLAMKAJJIKAYALU', 'BELLAMKOMMULU',
       'BELLAMMITAICHAKKIES', 'BELLAMMITAIUNDALU',
       'BELLAMMITHAIUNDALUGHEE', 'BELLAMNUVVULUUNDALU',
       'BELLAMPUTHAREKULU', 'BELLAMSALIVIDI', 'BELLAMSUNNUNADALU',
       'BENGALIITEM', 'BLACKFORESTPASTRY01PI',
       'BLACKFORESTPASTRYEGGR01PI', 'BOBBATLU', 'BOONDILADDU',
       'BOONDILADDU(GHEE)', 'BOONDILADDUSUGARFR

In [22]:

def plot_monthly_price_scatter(df, item_name):
    # Step 1: Filter for the selected item
    item_df = df[df['Corrected_Item_Name'] == item_name].copy()

    # Step 2: Ensure Date is datetime
    item_df['Date'] = pd.to_datetime(item_df['Date'])

    # Step 3: Extract month for x-axis
    item_df['Week'] = item_df['Date'].dt.to_period('W').dt.to_timestamp()

    # Step 4: Calculate frequency of each (Month, price_perkg)
    freq_df = (
        item_df.groupby(['Week', 'price_perkg'])
        .size()
        .reset_index(name='Frequency')
    )

    # Step 5: Scatter plot with different markers for each price_perkg
    fig = px.scatter(
        freq_df,
        x='Week',
        y='price_perkg',
        size='Frequency',
        color='price_perkg',
        title=f"Weekly Price Trends for {item_name}",
        labels={
            'Week': 'Week',
            'price_perkg': 'Price per Kg',
            'Frequency': 'Frequency Count'
        }
    )

    fig.show()


#df_mode_filtered


#df1 has multiple prices
#df2 has 'mode' pricelist
plot_monthly_price_scatter(weekly_sales_mode_price, 'KAJUPAKODI0200G')

In [11]:
plot_monthly_price_scatter(weekly_sales_table, 'KAJUPAKODI0200G')

In [12]:
from pygam import s, ExpectileGAM
import matplotlib.pyplot as plt
import numpy as np

>PRICE OPTIMISAATION & VISUALISATION

In [13]:
y=weekly_sales_mode_price['Item_Name'].unique().tolist()
y

['AGRAMIXTURE',
 'AGRAPETA',
 'AGRASEV',
 'ALMOND(BADAM)',
 'ALMONDBOUKEJ',
 'ALMONDTART',
 'ALOOBUJIYA',
 'ALOOLAYS',
 'ALOOMASALACHIPS',
 'AMLAMINI01PA',
 'ANJEERROLL',
 'ANJEERSANDWICH',
 'ANNAMAYYALADDU',
 'ANZEERTART',
 'ARISELUGHEE',
 'ATUKULU(KARAM)',
 'ATUKULU(SALT)',
 'ATUKULUSEV',
 'AYAIBURELU',
 'AZMEERKALAKAND',
 'AZMEERKALAKAND(SUGARFREE)',
 'BADAMHALWA',
 'BADAMLACHHA',
 'BADAMMYSOREPAK',
 'BADUSHAGHEE',
 'BADUSHAOIL',
 'BAKLAVA',
 'BANDARHALWA',
 'BANDARLADDU',
 'BASANTHI',
 'BATHISASOANPAPIDI',
 'BEACHBADAMBURFI',
 'BELLAM MITHAI UNDALU',
 'BELLAMGAVVALU',
 'BELLAMJELABI',
 'BELLAMKAJJIKAYALU',
 'BELLAMKOMMULU',
 'BELLAMMITAICHAKKIES',
 'BELLAMMITAIUNDALU',
 'BELLAMMITHAIUNDALUGHEE',
 'BELLAMNUVVULUUNDALU',
 'BELLAMPUTHAREKULU',
 'BELLAMSALIVIDI',
 'BELLAMSUNNUNADALU',
 'BENGALIITEM',
 'BLACKFORESTPASTRY01PI',
 'BLACKFORESTPASTRYEGGR01PI',
 'BOBBATLU',
 'BOONDILADDU',
 'BOONDILADDU(GHEE)',
 'BOONDILADDUSUGARFREE',
 'BORNVITAMYSOREPAK',
 'BORNVITASOANPAPDI',
 'BREADHALWA

In [14]:
from pygam import ExpectileGAM, s
from scipy.stats import zscore
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

def quantile_gam_by_dayfest(
    df, item_name, quantiles=[0.15, 0.5, 0.85], z_thresh=3, Dayfest=None
):
    # Step 1: Setup data groups
    if Dayfest is None or not Dayfest:
        Dayfest = ['All']  # special flag

    if len(Dayfest) > 4:
        raise ValueError("⚠️ Maximum 4 groups (DayFestGroup values) allowed.")

    # 2x2 grid layout
    rows = 2 if len(Dayfest) > 2 else 1
    cols = 2 if len(Dayfest) > 1 else 1

    fig = make_subplots(
        rows=rows,
        cols=cols,
        subplot_titles=[f"{grp}" for grp in Dayfest],
        shared_yaxes=True
    )

    results = {}

    for idx, group in enumerate(Dayfest):
        row = idx // 2 + 1
        col = idx % 2 + 1

        if group == 'All':
            df_temp = df[df['Corrected_Item_Name'] == item_name].copy()
        else:
            df_temp = df[
                (df['Corrected_Item_Name'] == item_name) & (df['DayFestGroup'] == group)
            ].copy()

        if df_temp.empty:
            print(f"⚠️ No data for group: {group}")
            continue

        df_temp['z_score'] = zscore(df_temp['price_perkg'])

        df_filtered = df_temp[(df_temp['z_score'] >= -z_thresh) & zscore(df_temp['Weekly_Weight'] <z_thresh)]

        x = df_filtered['price_perkg'].values.reshape(-1, 1)
        y = df_filtered['Weekly_Weight'].values

        if len(x) < 10:
            print(f"⚠️ Not enough data for: {group}")
            continue

        XX = np.linspace(x.min(), x.max(), 1000).reshape(-1, 1)

        # Scatter plot
        fig.add_trace(go.Scatter(
            x=x.flatten(), y=y,
            mode='markers',
            name='Data Points' if idx == 0 else None,
            marker=dict(size=6, opacity=0.5),
            hovertemplate='Price: ₹%{x:.2f}<br>Qty: %{y:.2f}',
            showlegend=(idx == 0)
        ), row=row, col=col)

        optimal_price, optimal_qty = None, None
        gam_models = {}

        for q in quantiles:
            gam = ExpectileGAM(s(0), expectile=q).fit(x, y)
            y_pred = gam.predict(XX)
            gam_models[q] = gam

            fig.add_trace(go.Scatter(
                x=XX.flatten(), y=y_pred,
                mode='lines',
                name=f'{int(q*100)}th Quantile' if idx == 0 else None,
                hovertemplate='Price: ₹%{x:.2f}<br>Pred Qty: %{y:.2f}',
                showlegend=(idx == 0)
            ), row=row, col=col)

            if q == 0.5:
                max_idx = np.argmax(y_pred)
                optimal_price = XX[max_idx][0]
                optimal_qty = y_pred[max_idx]

                fig.add_trace(go.Scatter(
                    x=[optimal_price], y=[optimal_qty],
                    mode='markers+text',
                    name='Optimal Price' if idx == 0 else None,
                    marker=dict(color='red', size=10),
                    text=[f'₹{optimal_price:.2f}'],
                    textposition='top center',
                    hovertemplate='Optimal Price: ₹%{x:.2f}<br>Qty: %{y:.2f}',
                    showlegend=(idx == 0)
                ), row=row, col=col)

        results[group] = {
            "optimal_price": round(optimal_price, 2) if optimal_price else None,
            "filtered_data": df_filtered,
            "gam_models": gam_models
        }

    fig.update_layout(
        height=600 if rows == 1 else 900,
        width=1000,
        title=f'Quantile GAMs: Price Elasticity for "{item_name}"',
        xaxis_title='Price per Kg (₹)',
        yaxis_title='Weekly Quantity (Kg)',
        template='plotly_white',
        legend_title='GAM Curves'
    )

    fig.show()
    return results


In [15]:
z_thresh = 3
weekly_prices_round = weekly_sales_table.copy()
weekly_prices_round['price_perkg'] = weekly_sales_table['Weekly_Weight'].round(0)

weekly_prices_round['z_score'] = zscore(weekly_prices_round['Weekly_Weight'])

weekly_prices_round = weekly_prices_round[(weekly_prices_round['z_score'] >= -z_thresh) & zscore(weekly_prices_round['Weekly_Weight'] <z_thresh)]

weekly_prices_round

,Date,Item_Name,Corrected_Item_Name,Outlet,price_perkg,Week,Weekly_Weight,Weekly_Amount,DayFestGroup,Event,z_score
0,2021-07-08,AGRAMIXTURE,AGRAMIXTURE0250G,ASILMETTA OUTLET,1.0,2021-07-05,0.75,270.0129,Wkdy - Non-Festive,No Event,-0.076403
1,2021-07-09,AGRAMIXTURE,AGRAMIXTURE0250G,ASILMETTA OUTLET,0.0,2021-07-05,0.50,179.9986,Wknd - Non-Festive,No Event,-0.089776
2,2021-08-11,AGRAMIXTURE,AGRAMIXTURE0250G,ASILMETTA OUTLET,1.0,2021-08-09,1.00,360.0072,Wkdy - Non-Festive,Independence Day,-0.063030
3,2021-08-19,AGRAMIXTURE,AGRAMIXTURE0250G,ASILMETTA OUTLET,0.0,2021-08-16,0.25,90.0043,Wkdy - Non-Festive,Raksha Bandhan,-0.103149
4,2021-08-21,AGRAMIXTURE,AGRAMIXTURE0250G,ASILMETTA OUTLET,13.0,2021-08-16,12.75,4563.0562,Wknd - Festive,Raksha Bandhan,0.565506
...,...,...,...,...,...,...,...,...,...,...,...
781799,2022-07-21,YELLOWSOANPAPIDI(SUGARFREE),YELLOWSOANPAPIDI(SUGARFREE)0500G,SIRIPURAM,0.0,2022-07-18,0.50,380.0048,Wkdy - Non-Festive,No Event,-0.089776
781800,2022-07-28,YELLOWSOANPAPIDI(SUGARFREE),YELLOWSOANPAPIDI(SUGARFREE)0250G,SIRIPURAM,0.0,2022-07-25,0.50,380.0048,Wkdy - Non-Festive,No Event,-0.089776
781801,2022-12-20,YELLOWSOANPAPIDI(SUGARFREE),YELLOWSOANPAPIDI(SUGARFREE)1000G,SIRIPURAM,1.0,2022-12-19,1.00,759.9995,Wkdy - Non-Festive,Christmas,-0.063030
781802,2023-05-21,YELLOWSOANPAPIDI(SUGARFREE),YELLOWSOANPAPIDI(SUGARFREE)0250G,SIRIPURAM,0.0,2023-05-15,0.50,380.0048,Wknd - Festive,No Event,-0.089776


In [19]:
weekly_prices_round['Corrected_Item_Name'].sample(n=10)	

217248    DRYFRUITBESARLADDU0500G
671148       SADAKAJJIKAYALU0250G
92209      BELLAMPUTHAREKULU0500G
582209          PANEERJALEBI0500G
571832         ONIONCHEKKALU0200G
126224           BOONDILADDU0250G
158042                BURELU0250G
758363         WHITEKALAKAND0500G
451046       KARAMPOWDER0050G01PA
435084             KALAJAMUN1000G
Name: Corrected_Item_Name, dtype: object

In [23]:
#['Wkdy - Non-Festive', 'Wknd - Non-Festive', 'Wkdy - Festive','Wknd - Festive']

# Single group
res1 = quantile_gam_by_dayfest(df=weekly_sales_table, z_thresh=3, item_name='GHEEMIXEDSWEETS1000G')

# All groups
res2 = quantile_gam_by_dayfest(df=weekly_sales_table,z_thresh=3, item_name='KALAJAMUN1000G',
                                Dayfest=['Wkdy - Festive', 'Wknd - Festive', 'Wkdy - Non-Festive', 'Wknd - Non-Festive'])
# All groups
res2 = quantile_gam_by_dayfest(df=weekly_sales_table,z_thresh=3, item_name='GHEEMIXEDSWEETS0500G',
                                Dayfest=['Wkdy - Festive', 'Wknd - Festive', 'Wkdy - Non-Festive', 'Wknd - Non-Festive'])
